# RNA pocket detection and ligand docking with BioBB, fpocket and AutoDock Vina

**Reproducible, local-file workflow** adapted from the original Oxford RNA docking notebook (PDB **7LYJ**).

The notebook performs: **PDB preparation → fpocket pockets → pocket selection → docking box → ligand preparation → RNA receptor PDBQT → Vina → optional multi-ligand screening**.

> **No ZINC20/ZINC22 downloads.** All receptor PDBs and ligand SDFs are read from `data/`. This avoids ZINC CAPTCHA pages and makes the workflow suitable for a GitHub repository. See `README.md` for installation, folder layout, and data provenance. Computational docking scores are screening metrics, not measured binding affinities.

## 0. Prepare the project

1. Put your downloaded RNA PDB at `data/structures/7LYJ.pdb`.
2. Put your downloaded SDFs at `data/ligands/ZINC000003830390.sdf`, or keep the historical naming `data/ligands/tempZINC000003830390.sdf`.
3. Optionally put `Integrated.csv` under `data/` with a `Column1` header and one ligand ID per row.
4. Install the software using `conda env create -f environment.yml`, then `conda activate molecular-jupyter`, followed by `jupyter lab`.

**Google Colab:** Opening a notebook from GitHub does not automatically clone its input files or install the external binaries. In Colab, set `GITHUB_REPO_URL` below to your public repository so that the notebook clones its files. The native BioBB/`fpocket`/`obabel`/`vina` dependencies still need a compatible installation; local Conda is the tested workflow target. No downloads occur unless you explicitly provide a repository URL.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import shutil

# Leave empty for local JupyterLab. On Google Colab, set your public GitHub URL.
GITHUB_REPO_URL = ""    # Example: "https://github.com/USERNAME/rna-ligand-docking.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and GITHUB_REPO_URL:
    repository_name = GITHUB_REPO_URL.rstrip("/").removesuffix(".git").split("/")[-1]
    clone_dir = Path("/content") / repository_name
    if not (clone_dir / "docking_project.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(clone_dir)], check=True)
    PROJECT_ROOT = clone_dir
else:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, Path('/content/rna-ligand-docking')]
    PROJECT_ROOT = next((p for p in candidates if (p / 'docking_project.py').is_file()), cwd)

if not (PROJECT_ROOT / 'docking_project.py').is_file():
    raise FileNotFoundError(
        'Project module not found. Start Jupyter in the repository, or set GITHUB_REPO_URL on Colab.'
    )
sys.path.insert(0, str(PROJECT_ROOT))
from docking_project import (find_sdf, discover_ligands, validate_sdf,
                             validate_pdbqt, best_affinity, run_batch, VINA_DEFAULTS)

DATA = PROJECT_ROOT / 'data'
RESULTS = PROJECT_ROOT / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)
print('Repository:', PROJECT_ROOT)
print('In Colab:', IN_COLAB)

## 1. Set the input files and protocol parameters

Change only these variables to use another receptor, ligand, or pocket. `PRIMARY_LIGAND_ID` identifies a file already downloaded locally; it is **not** queried from ZINC. The pocket filter and box offset retain the original notebook's settings.

In [ ]:
PDB_CODE = '7LYJ'
PRIMARY_LIGAND_ID = 'ZINC000003830390'
RAW_PDB = DATA / 'structures' / f'{PDB_CODE}.pdb'
LIGAND_DIR = DATA / 'ligands'
INTEGRATED_CSV = DATA / 'Integrated.csv'  # optional; header must be Column1

PH = 7.4
POCKET_VOLUME_RANGE = [400, 2000]
SELECTED_POCKET = 1  # change after reviewing the pocket summary
BOX_OFFSET = 12
VINA_PROPERTIES = dict(VINA_DEFAULTS)

print('Receptor:', RAW_PDB)
print('Primary SDF:', LIGAND_DIR / f'{PRIMARY_LIGAND_ID}.sdf')

In [ ]:
required_bins = ('fpocket', 'obabel', 'vina', 'check_structure')
missing_bins = [name for name in required_bins if shutil.which(name) is None]
if missing_bins:
    raise RuntimeError('Missing executables on PATH: ' + ', '.join(missing_bins) +
                       '. Install the Conda environment from environment.yml.')
if not RAW_PDB.is_file():
    raise FileNotFoundError(f'Missing receptor: {RAW_PDB}. See data/structures/README.md')
primary_sdf = find_sdf(PRIMARY_LIGAND_ID, LIGAND_DIR)
validate_sdf(primary_sdf)  # also rejects a cached ZINC CAPTCHA page
print('Input files and required executables found.')

## 2. Prepare the RNA structure with PDBFixer

This follows the original notebook: remove heterogens (including waters/ions), avoid rebuilding absent terminal residues, fill missing atoms, and add hydrogens at nominal pH 7.4. **For RNA docking, inspect this structure carefully:** removing Mg²⁺ or other structural species can change the pocket, and RNA protonation/charge models need independent chemical validation.

In [ ]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

processed_pdb = RESULTS / 'processed.pdb'
fixer = PDBFixer(filename=str(RAW_PDB))
fixer.removeHeterogens(keepWater=False)
fixer.findMissingResidues()
chains = list(fixer.topology.chains())
for key in list(fixer.missingResidues):
    chain = chains[key[0]]
    if key[1] in (0, len(list(chain.residues()))):
        del fixer.missingResidues[key]
fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(PH)
with processed_pdb.open('w') as handle:
    PDBFile.writeFile(fixer.topology, fixer.positions, handle, keepIds=True)
assert processed_pdb.stat().st_size > 0
print('Prepared RNA:', processed_pdb)

In [ ]:
# Optional visualization: nglview frontend must work in your JupyterLab session.
import nglview as nv
view = nv.show_structure_file(str(processed_pdb))
view.clear_representations()
view.add_cartoon(color='lightblue')
view.center()
display(view)

## 3. Detect and filter pockets with fpocket

The original settings are sphere radius 3–6, minimum 35 spheres, and pocket volume 400–2000. Use the printed JSON summary and the optional NGL viewer to select a pocket. For different RNA conformations, pocket IDs and locations may change.

In [ ]:
import json
from biobb_vs.fpocket.fpocket_run import fpocket_run
from biobb_vs.fpocket.fpocket_filter import fpocket_filter

all_pockets = RESULTS / 'fpocket_all_pockets.zip'
pocket_summary = RESULTS / 'fpocket_summary.json'
filtered_pockets = RESULTS / 'fpocket_filtered.zip'
rc = fpocket_run(input_pdb_path=str(processed_pdb),
                 output_pockets_zip=str(all_pockets), output_summary=str(pocket_summary),
                 properties={'min_radius': 3, 'max_radius': 6, 'num_spheres': 35})
if rc != 0 or not pocket_summary.exists():
    raise RuntimeError(f'fpocket failed with exit code {rc}')
with pocket_summary.open() as f:
    print(json.dumps(json.load(f), indent=2)[:15000])
rc = fpocket_filter(input_pockets_zip=str(all_pockets),
                    input_summary=str(pocket_summary),
                    output_filter_pockets_zip=str(filtered_pockets),
                    properties={'volume': POCKET_VOLUME_RANGE})
if rc != 0 or not filtered_pockets.exists():
    raise RuntimeError('Pocket filtering failed or produced no output. Adjust POCKET_VOLUME_RANGE.')

In [ ]:
# Display filtered pocket IDs and, optionally, view pocket surfaces.
import zipfile
import re
pockets_dir = RESULTS / 'pockets'
pockets_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(filtered_pockets) as archive:
    pocket_files = [name for name in archive.namelist() if name.lower().endswith(('.pdb','.pqr'))]
    if not pocket_files:
        raise RuntimeError('No pockets passed the volume filter')
    archive.extractall(pockets_dir)
pocket_ids = sorted({int(m.group(1)) for name in pocket_files
                     if (m := re.search(r'pocket(\d+)_', Path(name).name))})
print('Available filtered pocket IDs:', pocket_ids)
print('Configured SELECTED_POCKET:', SELECTED_POCKET)
if SELECTED_POCKET not in pocket_ids:
    print('WARNING: Selected pocket not present; change SELECTED_POCKET in Section 1.')
# Visual inspection; turn off if large pockets make rendering slow.
pocket_view = nv.show_structure_file(str(processed_pdb))
pocket_view.clear_representations()
pocket_view.add_cartoon(color='silver', opacity=0.5)
for pocket_file in sorted(pockets_dir.rglob('pocket*_atm.pdb'))[:20]:
    pocket_view.add_component(str(pocket_file))
pocket_view.center()
display(pocket_view)

## 4. Select one pocket and generate the docking box

Set `SELECTED_POCKET` above, then rerun this cell. The `box.pdb` file is passed directly to BioBB's Vina wrapper, which computes the center and dimensions. This preserves the original approach instead of hard-coding one receptor's box coordinates.

In [ ]:
from biobb_vs.fpocket.fpocket_select import fpocket_select
from biobb_vs.utils.box import box

pocket_cavity = RESULTS / 'fpocket_cavity.pdb'
pocket_points = RESULTS / 'fpocket_pocket.pqr'
docking_box = RESULTS / 'box.pdb'
if SELECTED_POCKET not in pocket_ids:
    raise ValueError(f'Choose a pocket from {pocket_ids}')
rc = fpocket_select(input_pockets_zip=str(filtered_pockets),
                   output_pocket_pdb=str(pocket_cavity),
                   output_pocket_pqr=str(pocket_points),
                   properties={'pocket': int(SELECTED_POCKET)})
if rc != 0 or not pocket_points.exists():
    raise RuntimeError('fpocket_select failed')
rc = box(input_pdb_path=str(pocket_points), output_pdb_path=str(docking_box),
         properties={'offset': BOX_OFFSET, 'box_coordinates': True})
if rc != 0 or not docking_box.exists():
    raise RuntimeError('Docking-box generation failed')
print('Docking box:', docking_box)

In [ ]:
# Optional visual inspection of box pseudo-atoms and the RNA.
box_view = nv.show_structure_file(str(processed_pdb))
box_view.clear_representations()
box_view.add_cartoon(color='gray', opacity=0.5)
box_view.add_component(str(docking_box))
box_view.center()
display(box_view)

## 5. Prepare one ligand using existing SDF coordinates

Converts an existing **single-molecule** SDF to PDB and then PDBQT, using Open Babel through BioBB. We do not generate a new conformer or contact any external server. Check protonation, bond orders, stereochemistry, and charges before interpreting Vina results.

In [ ]:
from biobb_chemistry.babelm.babel_convert import babel_convert

single_dir = RESULTS / 'single'
single_dir.mkdir(parents=True, exist_ok=True)
ligand_pdb = single_dir / 'ligand.pdb'
ligand_pdbqt = single_dir / 'prep_ligand.pdbqt'
for input_path, output_path, fmt_in, fmt_out in [
    (primary_sdf, ligand_pdb, 'sdf', 'pdb'),
    (ligand_pdb, ligand_pdbqt, 'pdb', 'pdbqt'),
]:
    rc = babel_convert(input_path=str(input_path), output_path=str(output_path),
                       properties={'input_format': fmt_in,
                                   'output_format': fmt_out, 'binary_path': 'obabel'})
    if rc != 0 or not output_path.exists():
        raise RuntimeError(f'Ligand conversion failed: {fmt_in} -> {fmt_out}')
print('Ligand PDBQT:', ligand_pdbqt)

## 6. Prepare and check the RNA receptor PDBQT

The original notebook uses `biobb_structure_utils.str_check_add_hydrogens` with `charges=True`. We use `mode='auto'` because the `ph` mode was rejected by the installed `check_structure` utility in the prior run. **Successful file creation is not proof of chemically suitable RNA atom types/charges.** The check below verifies only PDBQT formatting.

In [ ]:
from biobb_structure_utils.utils.str_check_add_hydrogens import str_check_add_hydrogens

receptor_pdbqt = RESULTS / 'prep_receptor.pdbqt'
rc = str_check_add_hydrogens(input_structure_path=str(processed_pdb),
                             output_structure_path=str(receptor_pdbqt),
                             properties={'charges': True, 'mode': 'auto'})
if rc != 0:
    raise RuntimeError(f'Receptor preparation failed with exit code {rc}')
print('PDBQT format check:', validate_pdbqt(receptor_pdbqt))
print('Receptor PDBQT:', receptor_pdbqt)

## 7. Run a single AutoDock Vina calculation

Defaults match the earlier BioBB notebook: 1 CPU, exhaustiveness 8, 9 modes, minimum RMSD 1, energy range 3. This workflow does **not** set a Vina random seed; scores/poses can vary between runs or Vina versions.

In [ ]:
from biobb_vs.vina.autodock_vina_run import autodock_vina_run

single_poses = single_dir / 'output_vina.pdbqt'
single_log = single_dir / 'output_vina.log'
rc = autodock_vina_run(input_ligand_pdbqt_path=str(ligand_pdbqt),
                       input_receptor_pdbqt_path=str(receptor_pdbqt),
                       input_box_path=str(docking_box),
                       output_pdbqt_path=str(single_poses),
                       output_log_path=str(single_log),
                       properties=VINA_PROPERTIES)
if rc != 0 or not single_poses.exists():
    raise RuntimeError(f'Vina docking failed with exit code {rc}; check {single_log}')
print('Best Vina affinity (kcal/mol):', best_affinity(single_poses))
print('Pose file:', single_poses)

In [ ]:
# Optional: view docked poses (notebook may be slow when all poses are shown).
pose_view = nv.show_structure_file(str(single_poses))
pose_view.add_ball_and_stick()
pose_view.center()
display(pose_view)

## 8. Batch virtual screening from *local* SDF files

If `data/Integrated.csv` exists, the `Column1` IDs determine which SDFs are used; otherwise the helper discovers all `*.sdf` in `data/ligands/`. Filenames `tempZINC...sdf` are supported for compatibility with the old code.

**Start with `MAX_LIGANDS=5`** to check the workflow, then set `None` for all ligands. The default `MAX_WORKERS=1` avoids the notebook's previous `multiprocessing.Pool` error; with the module-level worker you can try `3` once serial runs work. Errors are stored in the result CSV rather than being hidden by `MaybeEncodingError`. Reruns skip ligands with a valid existing docking score.

In [ ]:
MAX_LIGANDS = 5  # set to None to screen every local ligand
MAX_WORKERS = 1  # increase to 3 after successful serial docking

ligand_ids = discover_ligands(LIGAND_DIR, INTEGRATED_CSV, csv_column='Column1')
if not ligand_ids:
    raise RuntimeError('No ligands found. Copy local SDF files into data/ligands/.')
if MAX_LIGANDS is not None:
    ligand_ids = ligand_ids[:MAX_LIGANDS]
print(f'About to screen {len(ligand_ids)} ligands:', ligand_ids[:10])

In [ ]:
batch_dir = RESULTS / 'batch'
records = run_batch(ligand_ids, ligand_dir=LIGAND_DIR,
                    receptor=receptor_pdbqt, box=docking_box,
                    out_dir=batch_dir, max_workers=MAX_WORKERS,
                    vina_props=VINA_PROPERTIES, skip_existing=True)
print('Saved results:', batch_dir / 'results.csv')
print('Successful/skipped:', sum(r['status'] in ('ok', 'skipped') for r in records))
print('Errors:', sum(r['status'] == 'error' for r in records))

In [ ]:
import pandas as pd
summary = pd.read_csv(batch_dir / 'results.csv')
display(summary.head(20))
if (summary['status'] == 'error').any():
    print('Failures (see individual vina.log files for command output):')
    display(summary.loc[summary['status'] == 'error', ['ligand_id', 'error']].head(20))

## 9. Reproducibility and limitations

- **Input coordinates:** use the original downloaded PDB and SDF files to reproduce prior work; substituting new conformers or protonation states changes the problem.
- **RNA chemistry:** this tutorial-like pipeline does not validate receptor charge assignment, RNA flexibility, metal-ion dependence, or the suitability of Vina's protein-oriented scoring for RNA.
- **Pocket choices:** pocket numbering and box location depend on the receptor coordinates and fpocket settings.
- **Versions:** record your installed executable versions (`vina --version`, `obabel -V`, `fpocket -h`) and Conda environment (`conda env export --from-history`).
- **Results:** generated files go to `results/` (excluded from Git), and `results/batch/results.csv` stores per-ligand status and best docking score.

For a runnable command-line input check, use `python scripts/validate_inputs.py` from the project root. GitHub Actions only tests local-file helper functions, not expensive docking or correctness of RNA force-field choices.